In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm

*setup*

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

split_output = r"C:\Users\Sina_Si1\Desktop\Project Code\Phase 2\PV_Split"
train_dir = os.path.join(split_output, "train")
val_dir = os.path.join(split_output, "val")
test_dir = os.path.join(split_output, "test")

*Transforms and Dataloader*

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

train_data = datasets.ImageFolder(train_dir, transform=train_transforms)
val_data = datasets.ImageFolder(val_dir, transform=val_test_transforms)
test_data = datasets.ImageFolder(test_dir, transform=val_test_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=4)

num_classes = len(train_data.classes)
print("Dataset ready with", len(train_data), "train,", len(val_data), "val,", len(test_data), "test samples.")


*Pics of Dataset*

In [ ]:
def show_random_12(train_dataset):
    indices = random.sample(range(len(train_dataset)), 12)
    fig, axes = plt.subplots(3, 4, figsize=(12,9))
    fig.suptitle("Random 12 Images from Train Dataset", fontsize=16)
    mean = np.array([0.485,0.456,0.406])
    std = np.array([0.229,0.224,0.225])
    for i, ax in enumerate(axes.flat):
        img, label = train_dataset[indices[i]]
        # img is normalized tensor -> undo normalization
        img = img.cpu().numpy().transpose((1,2,0))
        img = std * img + mean
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title(f"{train_dataset.classes[label]}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_random_12(train_data)

*Ensemble Function*

In [ ]:
def ensemble_from_checkpoints(model_classes, checkpoint_paths, dataloader, device, num_classes, weights=None):
    models_list = []
    for model_class, ckpt_path in zip(model_classes, checkpoint_paths):
        # Initialize model
        model = model_class(num_classes=num_classes).to(device)
        # Load checkpoint
        checkpoint = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        models_list.append(model)

    if weights is None:
        weights = [1.0] * len(models_list)
    weights = np.array(weights) / np.sum(weights)

    all_labels, all_preds = [], []
    total_loss, correct, total = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Ensemble Inference"):
            inputs, labels = inputs.to(device), labels.to(device)
            ensemble_out = torch.zeros((inputs.size(0), num_classes), device=device)
            for model, w in zip(models_list, weights):
                ensemble_out += w * F.softmax(model(inputs), dim=1)

            preds = ensemble_out.argmax(dim=1)
            loss = criterion(ensemble_out, labels)

            total_loss += loss.item() * labels.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    avg_loss = total_loss / total
    accuracy = 100 * correct / total

    print(f"\nEnsemble Results (Weighted):")
    print(f"Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, digits=4))

    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(15, 15))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=ax, cmap="Blues", colorbar=True, xticks_rotation=90)
    plt.title("Confusion Matrix - Ensemble (Weighted)")
    plt.show()

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "report": classification_report(all_labels, all_preds, digits=4, output_dict=True),
        "confusion_matrix": cm
    }

**Run Ensemble**

*2 Network Ensemble*

In [ ]:
checkpoint_paths = [
    r"C:\Users\Sina_Si1\Desktop\Checkpoints/EfficientNetB0.pth",
    r"C:\Users\Sina_Si1\Desktop\Checkpoints/MobileNetV2.pth"
]

weights = [0.9942, 0.9867]  # Based on validation accuracy

model_classes = [
    lambda num_classes: models.efficientnet_b0(weights=None, num_classes=num_classes),
    lambda num_classes: models.mobilenet_v2(weights=None, num_classes=num_classes)
]

results = ensemble_from_checkpoints(
    model_classes=model_classes,
    checkpoint_paths=checkpoint_paths,
    dataloader=test_loader,
    device=device,
    num_classes=num_classes,
    weights=weights
)

*3 Network Ensemble*

In [ ]:
checkpoint_paths = [
    r"C:\Users\Sina_Si1\Desktop\Checkpoints/EfficientNetB0.pth",
    r"C:\Users\Sina_Si1\Desktop\Checkpoints/MobileNetV2.pth",
    r"C:\Users\Sina_Si1\Desktop\Checkpoints/ResNet50.pth"
]

weights = [0.9942, 0.9867, 0.9856]  # Based on validation accuracy

model_classes = [
    lambda num_classes: models.efficientnet_b0(weights=None, num_classes=num_classes),
    lambda num_classes: models.mobilenet_v2(weights=None, num_classes=num_classes),
    lambda num_classes: models.resnet50(weights=None, num_classes=num_classes)
]

results = ensemble_from_checkpoints(
    model_classes=model_classes,
    checkpoint_paths=checkpoint_paths,
    dataloader=test_loader,
    device=device,
    num_classes=num_classes,
    weights=weights
)